# Fase 6: Framework de Validação e Backtest Sem Overfitting (CPCV, DSR, PBO)

Este notebook demonstra o framework institucional de validação estatística de modelos:
1. **Combinatorial Purged Cross-Validation (CPCV)**: Divisão em $N$ partições com testes de todas as combinações $\binom{N}{k}$, aplicando *Purging* e *Embargo*.
2. **Deflated Sharpe Ratio (DSR)**: Ajuste do Sharpe considerando a assimetria, curtose e penalidade de seleção múltipla ($N_{\text{trials}}$).
3. **Probability of Backtest Overfitting (PBO)**: Probabilidade de que a melhor estratégia in-sample falhe fora de amostra.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import RandomForestClassifier

sys.path.insert(0, os.path.abspath('..'))

from src.validation.cpcv_evaluator import (
    CombinatorialPurgedKFold,
    probabilistic_sharpe_ratio,
    deflated_sharpe_ratio,
    compute_pbo,
    CPCVEvaluator,
)

In [2]:
# Simulação de Série Temporal de Features e Retornos
np.random.seed(42)
n_obs = 300
dates = pd.date_range("2024-01-01", periods=n_obs, freq="B")

f1 = np.random.normal(0, 1, n_obs)
f2 = np.random.normal(0, 1, n_obs)
f3 = np.random.normal(0, 1, n_obs)
X = pd.DataFrame({"f1": f1, "f2": f2, "f3": f3}, index=dates)

returns = pd.Series(0.012 * f1 - 0.008 * f2 + np.random.normal(0, 0.015, n_obs), index=dates)
y = pd.Series(np.where(returns > 0, 1, 0), index=dates)

models = {
    "Ridge_Alpha_1.0": RidgeClassifier(alpha=1.0),
    "RandomForest_D3": RandomForestClassifier(n_estimators=20, max_depth=3, random_state=42),
    "RandomForest_D5": RandomForestClassifier(n_estimators=30, max_depth=5, random_state=42),
}

# Avaliação CPCV com 5 partições e combinações de 2 (Total: C(5, 2) = 10 caminhos)
evaluator = CPCVEvaluator(n_splits=5, n_test_splits=2, pct_embargo=0.01)
val_results = evaluator.evaluate_model_family(models, X, y, returns)

print(val_results["report_markdown"])